# Автоэнкодер для поиска подозрительных наблюдений

## Цель

Этот ноутбук показывает второй нейросетевой блок проекта.

В отличие от предыдущего ноутбука, здесь нейросеть используется не для прямого прогноза скорости изменения береговой бровки. Здесь автоэнкодер применяется как инструмент диагностики данных: он помогает найти строки с нетипичными сочетаниями признаков.

Такие строки нельзя автоматически считать ошибочными. Их нужно рассматривать как кандидатов на ручную проверку: посмотреть участок, профиль, даты интервала, QC-пояснения и возможные проблемы исходных наблюдений.

Роль ноутбука в НИР:

- показать нейросетевой подход к статистической проверке данных;
- выделить потенциально нетипичные наблюдения;
- дополнить базовое моделирование отдельным блоком контроля качества.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from src.analysis.autoencoder_anomaly_detection import run_autoencoder_anomaly_detection

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
reports_dir = project_root / 'reports'


## Метод

Автоэнкодер — это нейросетевая модель, которая учится восстанавливать собственные входные данные.

В этом проекте используется упрощённая автоэнкодерная схема на основе `MLPRegressor` из `scikit-learn`. На вход модели подаётся подготовленная матрица признаков, и эта же матрица используется как целевое значение для восстановления.

Если строка хорошо соответствует общей структуре датасета, модель обычно восстанавливает её с небольшой ошибкой. Если строка содержит нетипичное сочетание признаков, ошибка восстановления может быть выше.

В модель не подаются целевая переменная, прямые метрики изменения береговой бровки, идентификаторы и служебные текстовые поля. Это сделано, чтобы автоэнкодер анализировал структуру признаков, а не получал готовый ответ или технические поля.

Порог подозрительности выбран как 95-й процентиль ошибки восстановления на обучающей части. Это означает, что в список для проверки попадают строки с наиболее высокой ошибкой восстановления.

In [ ]:
outputs = run_autoencoder_anomaly_detection(verbose=False)

pd.DataFrame({
    'показатель': [
        'обработано строк',
        'признаков до кодирования категорий',
        'признаков после предобработки',
        'строк в обучающей части для порога',
        'порог ошибки восстановления',
        'подозрительных строк',
    ],
    'значение': [
        outputs['n_rows'],
        outputs['n_features_raw'],
        outputs['n_features_encoded'],
        outputs['n_train'],
        round(outputs['threshold'], 6),
        outputs['n_anomalies'],
    ],
})

## Что считается подозрительным

Для каждой строки считается ошибка восстановления: средняя квадратичная разница между подготовленным вектором признаков и его восстановленной версией. Подозрительными считаются строки выше 95-го процентиля ошибки восстановления на обучающей части.

Такая строка не считается доказанной ошибкой. Она означает только, что сочетание признаков нетипично для текущей модели и требует предметной проверки.

In [ ]:
scores = pd.read_csv(reports_dir / 'tables' / 'autoencoder_anomaly_scores.csv')
top_anomalies = pd.read_csv(reports_dir / 'tables' / 'autoencoder_top_anomalies.csv')

display(Markdown((reports_dir / 'tables' / 'autoencoder_anomaly_summary.md').read_text(encoding='utf-8')))

## Результаты

В результате работы автоэнкодера формируются две основные таблицы и два графика.

Таблица `autoencoder_anomaly_scores.csv` содержит все строки с рассчитанной ошибкой восстановления и признаком подозрительности.

Таблица `autoencoder_top_anomalies.csv` содержит строки с наибольшей ошибкой восстановления. Именно её удобно использовать как короткий список для ручной проверки.

Первый график показывает распределение ошибок восстановления и выбранный порог. Он нужен для отчёта, потому что визуально показывает, какая часть наблюдений считается нетипичной.

Второй график показывает 25 строк с наибольшей ошибкой восстановления. Он полезен как техническая иллюстрация, но для основного текста отчёта лучше использовать его осторожно: он показывает кандидатов на проверку, а не доказанные ошибки.


In [ ]:
display(Image(filename=str(reports_dir / 'figures' / '04_autoencoder_reconstruction_error.png')))
display(Image(filename=str(reports_dir / 'figures' / '04_autoencoder_top_anomalies.png')))

In [ ]:
top_anomalies.head(25)

## Ограничения

Автоэнкодер не доказывает, что найденная строка является ошибкой. Он показывает только то, что сочетание признаков в этой строке плохо восстанавливается моделью и отличается от типичной структуры датасета.

Результат зависит от набора признаков, предобработки, масштаба числовых переменных, кодирования категориальных признаков и качества исходных данных.

Строки с высокой ошибкой восстановления нельзя автоматически удалять из датасета. Их нужно проверять вручную: смотреть участок, профиль, даты интервала, QC-пояснения, конфликтующие дубли и возможные ограничения по ветру или воде.

В рамках НИР этот ноутбук используется как демонстрация нейросетевого инструмента контроля качества данных.